# Ingesta de Datos: Indicadores Macroeconómicos

Este notebook se encarga de descargar, descomprimir y preprocesar los datos originales del Banco Mundial para estructurarlos en el formato requerido en la carpeta `data/in`.

In [7]:
import pandas as pd
import requests
import zipfile
import wbgapi as wb
import io
import os

TMP_DIR = './data/tmp'
IN_DIR = './data/in'

os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(IN_DIR, exist_ok=True)

# 1. Configuración de indicadores
# Mapeamos los códigos técnicos a nombres legibles para humanos
indicadores = {
    'NY.GDP.MKTP.CD': 'PIB_USD_Actual',
    'NY.GDP.MKTP.KD.ZG': 'Crecimiento_PIB_Anual',
    'NY.GDP.PCAP.CD': 'PIB_PerCapita_USD',
    'FP.CPI.TOTL.ZG': 'Inflacion_IPC',
    'FR.INR.RINR': 'Tasa_Interes_Real',
    'NE.EXP.GNFS.ZS': 'Exportaciones_pct_PIB',
    'BX.KLT.DINV.WD.GD.ZS': 'IED_Neta_pct_PIB',
    'BN.CAB.XOKA.GD.ZS': 'Cuenta_Corriente_pct_PIB',
    'SL.UEM.TOTL.ZS': 'Desempleo_Total_pct',
    'SL.TLF.CACT.ZS': 'Tasa_Participacion_Laboral'
}


In [8]:
def descargar_y_descomprimir_datos(url: str, nombre_archivo: str, directorio_destino: str = "data/tmp") -> None:
    """
    Descarga un archivo ZIP desde una URL, extrae específicamente el 
    archivo de datos principal y lo guarda con el nombre indicado por el usuario.
    
    Args:
        url (str): La URL del archivo .zip a descargar.
        nombre_archivo (str): Nombre deseado para el archivo final (ej. 'pib_usd_actual.csv').
        directorio_destino (str): El directorio local donde se guardará.
                                  Por defecto es 'data/tmp' dentro del proyecto.
    """
    
    # Crear los directorios (data y tmp) localmente si no existen
    os.makedirs(directorio_destino, exist_ok=True)
    
    # Asegurarnos de que el nombre indicado por el usuario tenga la extensión .csv
    if not nombre_archivo.endswith('.csv'):
        nombre_archivo += '.csv'
        
    ruta_zip = os.path.join(directorio_destino, "datos_temporales.zip")
    ruta_archivo_final = os.path.join(directorio_destino, nombre_archivo)
    
    print(f"Descargando datos en formato comprimido desde: {url}")
    
    # 1. Descargar el archivo ZIP y guardarlo temporalmente
    respuesta = requests.get(url, stream=True)
    respuesta.raise_for_status()
    
    with open(ruta_zip, 'wb') as archivo:
        for bloque in respuesta.iter_content(chunk_size=8192):
            if bloque:
                archivo.write(bloque)
                
    print("Descarga completada. Buscando y extrayendo archivo de datos principal...")
    
    # 2. Descomprimir y buscar el archivo principal de datos
    with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
        lista_archivos = zip_ref.namelist()
        archivo_objetivo = None
        
        # Buscar el archivo de datos principal (omitimos los de 'Metadata_')
        for nombre in lista_archivos:
            if nombre.endswith('.csv') and not nombre.startswith('Metadata'):
                archivo_objetivo = nombre
                break
                
        # Fallback: Extraer el primer CSV que encuentre
        if not archivo_objetivo:
            archivos_csv = [f for f in lista_archivos if f.endswith('.csv')]
            if archivos_csv:
                archivo_objetivo = archivos_csv[0]
            else:
                raise FileNotFoundError("No se encontró ningún archivo CSV dentro del .zip descargado.")
        
        # 3. Extraer los datos y guardándolos en la capeta 'data/tmp'
        contenido_csv = zip_ref.read(archivo_objetivo)
        
        with open(ruta_archivo_final, 'wb') as archivo_salida:
            archivo_salida.write(contenido_csv)
            
    # 4. Limpieza: Borrar el archivo .zip temporal porque ya no lo necesitamos
    if os.path.exists(ruta_zip):
        os.remove(ruta_zip)
        
    print(f"Archivo de datos extraído y listo en: {ruta_archivo_final}")


In [9]:
def procesar_archivo_wb(nombre_archivo_tmp: str, ruta_salida_in: str, nombre_valor: str = "valor", sep_salida: str =',') -> None:
    """
    Lee de forma genérica cualquier archivo CSV extraído del Banco Mundial, lo transforma a formato largo,
    lo cruza con el maestro de países para obtener metadatos geográficos y lo guarda filtrado y estructurado.
    
    Args:
        nombre_archivo_tmp (str): Nombre del archivo en la carpeta temporal (ej. 'global_inflation.csv').
        ruta_salida_in (str): Ruta destino del archivo procesado (ej. 'data/in/global_inflation_countries.csv').
        nombre_valor (str): Nombre para la columna numérica (ej. 'inflation_rate').
        sep_salida (str): El delimitador para guardar el archivo (usualmente ',' o ';').
    """
    ruta_tmp = f"data/tmp/{nombre_archivo_tmp}"
    ruta_maestro = "data/in/countries_master.csv"
    
    print(f"Limpiando y cruzando archivo: {ruta_tmp}...")
    
    # 1. Leer el archivo saltando las 4 filas de metadata del World Bank
    df_raw = pd.read_csv(ruta_tmp, skiprows=4)
    
    # Cargar archivo maestro (separado por ';')
    df_master = pd.read_csv(ruta_maestro, sep=';')
    
    # 2. Filtrar columnas no válidas
    columnas_validas = [col for col in df_raw.columns if not col.startswith('Unnamed')]
    df = df_raw[columnas_validas].copy()
    
    # 3. Transformación 'Melt' (apilando los años)
    df_melt = df.melt(
        id_vars=['Country Code', 'Country Name', 'Indicator Code', 'Indicator Name'],
        var_name='year',
        value_name=nombre_valor
    )
    
    # 4. Eliminar el Country Name del Banco mundial para utilizar el del archivo Maestro y evitar duplicados
    df_melt = df_melt.drop(columns=['Country Name'])
    
    # 5. Estandarizar identificadores a nuestro sistema
    df_melt = df_melt.rename(columns={
        'Country Code': 'country_code',
        'Indicator Code': 'indicator_code',
        'Indicator Name': 'indicator_name'
    })
    
    # Convertir llaves a MAYÚSCULAS para cruzar seguros
    df_melt['country_code'] = df_melt['country_code'].astype(str).str.upper()
    df_melt['indicator_code'] = df_melt['indicator_code'].astype(str).str.upper()
    df_melt['indicator_name'] = df_melt['indicator_name'].astype(str).str.upper()
    
    # 6. Limpieza de tipos de datos de años y montos numéricos
    df_melt['year'] = pd.to_numeric(df_melt['year'], errors='coerce')
    df_melt.dropna(subset=['year'], inplace=True)
    df_melt['year'] = df_melt['year'].astype(int)
    
    df_melt[nombre_valor] = pd.to_numeric(df_melt[nombre_valor], errors='coerce').fillna(0.0)
    
    df_merged = pd.merge(
        df_melt, 
        df_master, 
        on='country_code', 
        how='inner'
    )
    
    # 8. Ordenar y estructurar tabla final
    df_merged = df_merged.sort_values(by=['country_code', 'year']).reset_index(drop=True)
    
    # Estructuramos la misma salida que pediste
    orden_final = [
        'country_code', 'region_name', 'sub_region_name', 'intermediate_region', 
        'country_name', 'income_group', 'indicator_code', 'indicator_name', 'year', nombre_valor
    ]
    df_final = df_merged[orden_final]
    
    # 9. Guardar la información consolidada
    df_final.to_csv("data/in/"+ruta_salida_in, sep=sep_salida, index=False)
    print(f"Éxito: Archivo final procesado con la estructura maestra en: {ruta_salida_in}")


Descarga los datos de GDP

In [10]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/NY.GDP.MKTP.CD?downloadformat=csv",
                               "gdp_hist")

procesar_archivo_wb("gdp_hist.csv",
                    "countries_gdp.csv",
                    "total_gdp",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/NY.GDP.MKTP.CD?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/gdp_hist.csv
Limpiando y cruzando archivo: data/tmp/gdp_hist.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_gdp.csv


Descarga los datos de crecimiento PIB %

In [11]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/NY.GDP.MKTP.KD.ZG?downloadformat=csv",
                               "gdp_variation")

procesar_archivo_wb("gdp_variation.csv",
                    "countries_gdp_variation.csv",
                    "variacion",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/NY.GDP.MKTP.KD.ZG?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/gdp_variation.csv
Limpiando y cruzando archivo: data/tmp/gdp_variation.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_gdp_variation.csv


Descarga los datos del PIB Percapita

In [12]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/NY.GDP.PCAP.CD?downloadformat=csv",
                               "gdp_percapita_hist")

procesar_archivo_wb("gdp_percapita_hist.csv",
                    "countries_gdp_percapita.csv",
                    "total_gdp_percapita",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/NY.GDP.PCAP.CD?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/gdp_percapita_hist.csv
Limpiando y cruzando archivo: data/tmp/gdp_percapita_hist.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_gdp_percapita.csv


Descarga los datos de crecimiento PIB Percapita %

In [13]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/NY.GDP.PCAP.KD.ZG?downloadformat=csv",
                               "gdp_percapita_growth")

procesar_archivo_wb("gdp_percapita_growth.csv",
                    "countries_gdp_percapita_variation.csv",
                    "gdp_percapita_variation",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/NY.GDP.PCAP.KD.ZG?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/gdp_percapita_growth.csv
Limpiando y cruzando archivo: data/tmp/gdp_percapita_growth.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_gdp_percapita_variation.csv


descarga los datos de la inflacion

In [14]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/FP.CPI.TOTL.ZG?downloadformat=csv",
                               "inflacion")

"https://api.worldbank.org/v2/en/indicator/NY.GDP.DEFL.KD.ZG?downloadformat=csv"

procesar_archivo_wb("inflacion.csv",
                    "countries_inflation.csv",
                    "inflation",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/FP.CPI.TOTL.ZG?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/inflacion.csv
Limpiando y cruzando archivo: data/tmp/inflacion.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_inflation.csv


Descarga los datos de exportaciones como % del PIB


In [15]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/NE.EXP.GNFS.ZS?downloadformat=csv",
                               "exportaciones")

procesar_archivo_wb("exportaciones.csv",
                    "countries_exports.csv",
                    "exportacion",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/NE.EXP.GNFS.ZS?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/exportaciones.csv
Limpiando y cruzando archivo: data/tmp/exportaciones.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_exports.csv


Descarga los datos de importaciones como % PIB

In [16]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/NE.IMP.GNFS.ZS?downloadformat=csv",
                               "importaciones")

procesar_archivo_wb("importaciones.csv",
                    "countries_imports.csv",
                    "importacion",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/NE.IMP.GNFS.ZS?downloadformat=csv


Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/importaciones.csv
Limpiando y cruzando archivo: data/tmp/importaciones.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_imports.csv


Descarga los datos de Inversion extranjera


In [17]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/BX.KLT.DINV.WD.GD.ZS?downloadformat=csv",
                               "inversion_extranjera")

procesar_archivo_wb("inversion_extranjera.csv",
                    "countries_inversion_extranjera.csv",
                    "inversion_extranjera",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/BX.KLT.DINV.WD.GD.ZS?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/inversion_extranjera.csv
Limpiando y cruzando archivo: data/tmp/inversion_extranjera.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_inversion_extranjera.csv


Descarga los datos de reservas internacionales

In [18]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/FI.RES.TOTL.CD?downloadformat=csv",
                               "reservas_internacionales")

procesar_archivo_wb("reservas_internacionales.csv",
                    "countries_reservas_internacionales.csv",
                    "reservas_internacionales",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/FI.RES.TOTL.CD?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/reservas_internacionales.csv
Limpiando y cruzando archivo: data/tmp/reservas_internacionales.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_reservas_internacionales.csv


Descarga los datos de la deuda externa en USD

In [19]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/DT.DOD.DECT.CD?downloadformat=csv",
                               "deuda_externa")

procesar_archivo_wb("deuda_externa.csv",
                    "countries_deuda_externa.csv",
                    "deuda_externa",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/DT.DOD.DECT.CD?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/deuda_externa.csv
Limpiando y cruzando archivo: data/tmp/deuda_externa.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_deuda_externa.csv


Descarga los datos de desempleo

In [20]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/SL.UEM.TOTL.ZS?downloadformat=csv",
                               "desempleo")

procesar_archivo_wb("desempleo.csv",
                    "countries_desempleo.csv",
                    "desempleo",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/SL.UEM.TOTL.ZS?downloadformat=csv


Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/desempleo.csv
Limpiando y cruzando archivo: data/tmp/desempleo.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_desempleo.csv


Descarga los datos de población total

In [21]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/SP.POP.TOTL?downloadformat=csv",
                               "poblacion")

procesar_archivo_wb("poblacion.csv",
                    "countries_poblacion.csv",
                    "poblacion",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/SP.POP.TOTL?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/poblacion.csv
Limpiando y cruzando archivo: data/tmp/poblacion.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_poblacion.csv


Descarga los datos de cuenta corriente (% PIB)

In [22]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/BN.CAB.XOKA.GD.ZS?downloadformat=csv",
                               "cuenta_corriente")

procesar_archivo_wb("cuenta_corriente.csv",
                    "countries_cuenta_corriente.csv",
                    "cuenta_corriente",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/BN.CAB.XOKA.GD.ZS?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/cuenta_corriente.csv
Limpiando y cruzando archivo: data/tmp/cuenta_corriente.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_cuenta_corriente.csv


Descarga los datos de deuda pública (% PIB)

In [23]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/GC.DOD.TOTL.GD.ZS?downloadformat=csv",
                               "deuda_publica")

procesar_archivo_wb("deuda_publica.csv",
                    "countries_deuda_publica.csv",
                    "deuda_publica",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/GC.DOD.TOTL.GD.ZS?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/deuda_publica.csv
Limpiando y cruzando archivo: data/tmp/deuda_publica.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_deuda_publica.csv


Descarga los datos de ingresos tributarios (% PIB)

In [24]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/GC.TAX.TOTL.GD.ZS?downloadformat=csv",
                               "ingresos_tributarios")

procesar_archivo_wb("ingresos_tributarios.csv",
                    "countries_ingresos_tributarios.csv",
                    "ingresos_tributarios",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/GC.TAX.TOTL.GD.ZS?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/ingresos_tributarios.csv
Limpiando y cruzando archivo: data/tmp/ingresos_tributarios.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_ingresos_tributarios.csv


Descarga los datos del índice de Gini

In [25]:
descargar_y_descomprimir_datos("https://api.worldbank.org/v2/en/indicator/SI.POV.GINI?downloadformat=csv",
                               "gini")

procesar_archivo_wb("gini.csv",
                    "countries_gini.csv",
                    "gini",
                    ";")

Descargando datos en formato comprimido desde: https://api.worldbank.org/v2/en/indicator/SI.POV.GINI?downloadformat=csv
Descarga completada. Buscando y extrayendo archivo de datos principal...
Archivo de datos extraído y listo en: data/tmp/gini.csv
Limpiando y cruzando archivo: data/tmp/gini.csv...
Éxito: Archivo final procesado con la estructura maestra en: countries_gini.csv
